# A3b -- the vicinity pseudo-granule control

Answers **Reviewer #2's literal round-1 suggestion**, re-requested in round 2 and never run:

> "Or alternatively, define **pseudo-granules in the direct vicinity of actual granules** as a
> negative control."
>
> — and in round 2: *"A direct check at the detection step, such as the pseudo-granule negative
> control I suggested, would settle the question."*

### Why the vicinity, specifically

This is the sharpest available answer to *structured* ambient. A sphere placed a few um away
shares the granule's local environment -- the same cell density, the same plaque proximity, the
same brain region, the same z-plane -- but is not a detected aggregate. If granules were merely
locally-elevated ambient, the offset spheres would look the same. A tissue-wide random location
would not test that, because it also changes the neighbourhood.

### Three design decisions that decide whether this survives scrutiny

**Offsets are in-plane.** `layer_z` takes only **7 discrete values** (0, 1.5 ... 9.0 um), and both
`profile()` and `nc_filter()` query at `layer_z`. A 3D direction would push the centre off the grid
and break comparability with every real granule.

**The transcript-count comparison is a tautology, so it does not carry the argument.** `sphere_r`
is the *minimum enclosing radius* of the DBSCAN core points (`miniball.get_bounding_ball` on the
deduplicated cluster). It is an order statistic: the sphere is maximally dense by construction, so
any displaced copy at the same radius must capture no more. Reporting "pseudo-granules capture
fewer transcripts" and stopping would be reporting an algebraic identity. It is kept as
description; the decision rests on section 4.

**The detection predicate is eps-connectivity, not a count.** Three transcripts scattered across a
4 um sphere are not `eps = 1.5`-connected, so "contains >= 3 transcripts" badly overstates how
detectable a pseudo-granule is. Section 4 runs the actual `DBSCAN(eps, min_samples)` on the **same
seed gene**, restricted to the pseudo-sphere -- "any of 20 markers" is roughly 20x easier than
"3 Camk2a".

### On rejecting offsets that land on a real granule

Measured before deciding: per-plane 2D granule coverage is only **1.9% (WT) / 1.5% (AD)**
(`sum(pi r^2) / (tissue_area x 7 planes)`), so rejecting granule-overlapping offsets discards few
candidates and does **not** meaningfully bias the sample toward granule-sparse space. (A 3D
nearest-neighbour distance of 2.68 um looks alarming but is misleading -- it counts neighbours on
adjacent z-planes.) Both arms are reported anyway, and in the unrejected arm the fraction of
offsets that land on a real granule is itself a **result**: it measures how much of the immediate
vicinity is already called.

### Where this sits in the run

Needs `set1_*/sphere_dict.parquet` from the HGCC array (the seed gene section 4 matches on). Section 6 needs nothing from HGCC. Runs **once, top to bottom, with nothing to adjust** -- see the runbook in
`README.md`.

**Run this notebook from `R2_revision/ambient_controls/`, on the `mcDETECT-env` kernel.**

## 0. Setup

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
import warnings
import zlib
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
from scipy.spatial import cKDTree

sys.path.insert(0, str(Path.cwd()))          # run this notebook from ambient_controls/
import a3_config as C
import a3_common as A3

warnings.filterwarnings("ignore")
sc.settings.verbosity = 0

# -------------------- runtime gates -------------------- #
# THE DEFAULTS BELOW PRODUCE THE FINAL TABLES. Run top to bottom, once, and change nothing.
DRY_RUN = False          # True -> subsample for a cheap smoke pass over every cell. The tables a
                         #   dry run writes are NOT final; set it back to False and rerun.
VALIDATE = True          # section 7 correctness gates. On by default (A1/A2 have them off): they
                         #   are the LAST section, so every table is already written before they
                         #   run.
RUN_PREDICATE = True     # section 4 -- the load-bearing statistic
RUN_ROUGH_VARIANT = True # section 6 -- the zero-placement-bias variant, needs no HPC output
SEED = C.VICINITY_SEED

MAX_GRANULES = 200_000 if DRY_RUN else None   # None = all 681K/399K. Section 4's predicate is the
                                              # slow step, and it is what the whole control rests
                                              # on, so the final run must see every granule.

C.ensure_dirs()
OUT = C.A3B_DIR

print("writing to :", OUT)
print("arms       :", list(C.VICINITY_ARMS))
print("offsets    :", C.VICINITY_D, "um absolute +", C.VICINITY_D_RELATIVE, "x sphere_r")
print("predicate  : DBSCAN core point of the SEED gene inside the sphere"
      f" | eps = {C.EPS} | min_samples = {C.DETECT_KWARGS_FINE['minspl']}")
print("matched on :", C.VICINITY_MATCH_ON)

## 1. Source granules, tissue mask and density strata

The source set is the published Set 2. Three things are attached to each granule before any
offset is placed, because the pseudo-granule has to be matched on them:

* **`brain_area`** -- already on `granules.parquet` (nearest-spot label, `3_detection.py:88-95`).
* **local transcript-density quintile** on a 25 um lattice. This is the direct answer to "ambient
  is denser where cells are denser, or near plaques": if the pseudo-granule sits in the same
  density stratum as its source, a difference between them cannot be a density difference.
* **`seed_gene`** from the persisted `sphere_dict`. `granules.parquet["gene"]` is stale for merged
  granules -- `_remove_overlaps` updates only `sphere_x/y/z`, `layer_z` and `sphere_r` -- so the
  recorded gene cannot be trusted for the seed-matched predicate. Where the `sphere_dict` is not
  yet on disk, the recorded gene is used and the run is recorded in `source_summary.csv`.

In [ ]:
sources, masks, dens, nuc = {}, {}, {}, {}

for sample in C.SAMPLES:
    g = pd.read_parquet(C.mcdetect_granules_path(sample))
    if MAX_GRANULES and len(g) > MAX_GRANULES:
        g = g.sample(MAX_GRANULES, random_state=SEED).reset_index(drop=True)
    tx = A3.load_transcripts(sample, columns=["global_x", "global_y", "global_z", "target",
                                              "overlaps_nucleus"])

    mask, xb, yb = A3.tissue_mask(tx)
    qgrid, qxb, qyb = A3.density_quintiles(tx)

    # Nucleus point cloud, so offsets landing inside nuclei can actually be rejected -- both
    # arms are documented to do this and previously neither did.
    nuc_tx = tx[tx["overlaps_nucleus"] == 1]
    nuc[sample] = cKDTree(nuc_tx[["global_x", "global_y", "global_z"]].to_numpy(float))

    qi = np.clip(np.searchsorted(qxb, g["sphere_x"], side="right") - 1, 0, qgrid.shape[0] - 1)
    qj = np.clip(np.searchsorted(qyb, g["sphere_y"], side="right") - 1, 0, qgrid.shape[1] - 1)
    g["density_quintile"] = qgrid[qi, qj]

    # seed gene: prefer the persisted pre-merge dict; fall back to the stale column and say so
    sd_path = C.sphere_dict_path("set1", sample)
    if sd_path.exists():
        sd = pd.read_parquet(sd_path)
        tree = cKDTree(sd[["sphere_x", "sphere_y", "layer_z"]].to_numpy(float))
        dnn, nn = tree.query(g[["sphere_x", "sphere_y", "layer_z"]].to_numpy(float), k=1)
        g["seed_gene"] = sd["seed_gene"].to_numpy()[nn]
        g["seed_gene_d"] = dnn
        # The join is exact for unmerged granules; for MERGED ones the centre was refit by
        # miniball over the union, so the nearest pre-merge sphere is a guess. Cap it at the
        # granule's own radius and record the match rate rather than letting a distant sphere
        # silently donate its gene.
        bad = dnn > g["sphere_r"].to_numpy()
        g.loc[bad, "seed_gene"] = pd.NA
        g["seed_gene_source"] = "sphere_dict"
        print(f"[{sample}] seed gene matched for {(~bad).mean():.1%} of granules "
              f"(median NN distance {np.median(dnn):.3f} um)")
    else:
        g["seed_gene"] = g["gene"]
        g["seed_gene_d"] = np.nan
        g["seed_gene_source"] = "granules.parquet (STALE after merge_sphere)"
        print(f"[{sample}] WARNING: sphere_dict missing -- seed gene falls back to the stale column")

    g = g[g["seed_gene"].notna()].reset_index(drop=True)
    sources[sample] = g
    masks[sample] = (mask, xb, yb)
    dens[sample] = (qgrid, qxb, qyb)
    print(f"[{sample}] {len(g):,} source granules | "
          f"density strata {sorted(g['density_quintile'].unique())}")
    del tx

pd.DataFrame([dict(sample=s, n=len(v),
                   seed_gene_source=v["seed_gene_source"].iloc[0],
                   median_seed_gene_d=float(np.nanmedian(v["seed_gene_d"])))
              for s, v in sources.items()]).to_csv(OUT / "source_summary.csv", index=False)

## 2. Placement -- two arms, an offset sweep

`unrejected` rejects only in-nucleus and out-of-tissue offsets; `rejected` additionally rejects
offsets overlapping a real granule, using mcDETECT's **own** merge predicate and nothing stricter
(a pseudo-granule that merely intersects a real granule is no more inadmissible than two real
granules that intersect -- which they routinely do, since the merge rule requires centres within
`0.4*r`).

The offset sweep spans `2r`, `3r`, 5, 10, 20 and 50 um. The deliverable in section 5 is the
**shape** of the curve, not any single distance.

In [ ]:
pseudo_frames, place_status = [], []

for sample in C.SAMPLES:
    g = sources[sample]
    mask, xb, yb = masks[sample]
    # 3D, matching mcDETECT's merge predicate. A 2D tree rejects an offset for overlapping a
    # granule on ANY z-plane -- stricter than the rule the arm claims, and stricter than real
    # granules obey.
    real_tree = cKDTree(g[["sphere_x", "sphere_y", "layer_z"]].to_numpy(float))
    real_r = g["sphere_r"].to_numpy(float)

    offsets = ([("abs", d, np.full(len(g), float(d))) for d in C.VICINITY_D] +
               [("rel", m, m * g["sphere_r"].to_numpy(float)) for m in C.VICINITY_D_RELATIVE])

    for kind, label, dvec in offsets:
        for arm in C.VICINITY_ARMS:
            # crc32, not hash(): Python's string hash is salted per process, so the placement
            # would not reproduce across runs.
            seed = zlib.crc32(f"{C.VICINITY_SEED}|{sample}|{kind}|{label}|{arm}".encode())
            rng = np.random.default_rng(seed)
            ps = A3.place_vicinity_spheres(g, mask, xb, yb, dvec, arm,
                                           real_tree=real_tree, real_r=real_r,
                                           nuc_tree=nuc[sample], rng=rng)
            ps["sample"], ps["d_kind"], ps["d_label"] = sample, kind, label
            pseudo_frames.append(ps)
            place_status.append(dict(sample=sample, arm=arm, d_kind=kind, d_label=label,
                                     n=len(ps), n_accepted=int(ps["accepted"].sum()),
                                     frac_accepted=float(ps["accepted"].mean()),
                                     mean_retry=float(ps["n_retry"].mean())))
            print(f"[{sample}] {arm:<11} d={kind}:{label:<5} "
                  f"accepted {ps['accepted'].mean():.3f}")

pseudo = pd.concat(pseudo_frames, ignore_index=True)
status = pd.DataFrame(place_status)
status.to_csv(OUT / "placement_status.csv", index=False)
display(status)

In [ ]:
# How often does an offset land on a real granule? In the `unrejected` arm this is not a
# nuisance -- it is a RESULT: it measures how much of the immediate vicinity is already called,
# and it is the number that decides whether the `rejected` arm could have biased anything.
ov_rows = []
for (sample, kind, label), grp in pseudo[pseudo["arm"] == "unrejected"].groupby(
        ["sample", "d_kind", "d_label"], observed=True):
    acc = grp[grp["accepted"]]
    if len(acc) == 0:
        continue
    hit, cnt = A3.overlap_pairs(acc, sources[sample], criterion="intersect", z_col="layer_z")
    ov_rows.append(dict(sample=sample, d_kind=kind, d_label=label, n=len(acc),
                        frac_on_real_granule=float(hit.mean()),
                        mean_n_real_overlapped=float(cnt.mean())))
ov = pd.DataFrame(ov_rows)
ov.to_csv(OUT / "vicinity_overlap_with_real.csv", index=False)
display(ov)

## 3. Profiling -- descriptive only

Real and pseudo spheres profiled by **identical** code (`sphere_features.profile_spheres`, a
vectorised re-implementation of `mcDETECT.model.profile` -- same ball, same `layer_z` centre, same
counting rule).

Read this section as description, not as the result. As stated in the header, a matched-radius
count comparison is an algebraic identity: `sphere_r` is the minimum enclosing radius of the exact
core points, so the real sphere is maximally dense by construction. What is *not* pre-ordained is
the **shape** of the gap -- how the in-nucleus ratio, NC ratio and unique-gene count differ -- and
those are worth reporting side by side as a filter funnel.

In [ ]:
sf = A3._import_sphere_features()
prof_summ, prof_hist, funnel_rows = [], [], []

for sample in C.SAMPLES:
    tx = A3.load_transcripts(sample)
    genes = A3.load_genes(sample)
    nc_genes = A3.load_nc_genes(sample)

    arms = [("real", sources[sample])]
    for (kind, label, arm), grp in pseudo[pseudo["sample"] == sample].groupby(
            ["d_kind", "d_label", "arm"], observed=True):
        acc = grp[grp["accepted"]].reset_index(drop=True)
        if len(acc):
            arms.append((f"pseudo|{arm}|{kind}:{label}", acc))

    for name, frame in arms:
        feats, _X, _g = sf.profile_spheres(frame, sample=sample, transcripts=tx, genes=genes,
                                      marker_genes=C.SYN_GENES, nc_genes=nc_genes,
                                      verbose=False)
        for measure, col in [("n_total", "n_total"), ("n_marker", "n_marker"),
                             ("in_soma_ratio", "in_soma_ratio_all"),
                             ("nc_ratio", "nc_ratio_all")]:
            if col not in feats:
                continue
            s, h = A3.record_distribution(feats[col], measure, C.HIST_BINS[measure],
                                          sample=sample, arm=name)
            prof_summ.append(s)
            prof_hist.extend(h)
        # the same cascade applied to both sets -- a funnel, not a single number
        funnel_rows.append(dict(
            sample=sample, arm=name, n=len(feats),
            n_in_tissue=len(feats),
            n_extrasomatic=int((feats["in_soma_ratio_all"] < C.IN_SOMA_THR).sum()),
            n_nc_clean=int(((feats["in_soma_ratio_all"] < C.IN_SOMA_THR) &
                            ((feats["nc_ratio_all"] == 0) |
                             (feats["nc_ratio_all"] < C.NC_THR))).sum()),
        ))
    del tx

pd.DataFrame(prof_summ).to_csv(OUT / "profile_summary.csv", index=False)
pd.DataFrame(prof_hist).to_parquet(OUT / "profile_histogram.parquet", index=False)
fun = pd.DataFrame(funnel_rows)
fun.to_csv(OUT / "profile_funnel.csv", index=False)
display(fun)

## 4. The detection predicate -- would the detector have fired here?

This is what the reviewer's control actually asks, and it is the only statistic in this notebook
that is not pre-ordained by the geometry.

For each accepted pseudo-granule: pull the **seed gene's** transcripts within
`sphere_r + margin`, run `DBSCAN(eps=1.5, min_samples=3)` on them, and ask whether any core point
exists. Seed-matched, because "any of the 20 markers" is roughly 20x easier than "3 Camk2a", and
Camk2a alone is 47% of the published granules.

Reported against two references: the same predicate evaluated at the **real** granule (which must
fire, by construction -- a useful sanity check), and at **tissue-wide random locations**, which is
the asymptote the offset curve should approach as `d` grows.

In [ ]:
if RUN_PREDICATE:
    pred_rows = []
    for col in ("would_detect", "gene_missing"):
        pseudo[col] = False
    pseudo["n_local"] = 0
    for sample in C.SAMPLES:
        tx = A3.load_transcripts(sample, columns=["global_x", "global_y", "global_z", "target"])
        by_gene = {}
        for g in C.SYN_GENES:
            sub = tx[tx["target"] == g]
            if len(sub) == 0:
                continue
            coords = sub[["global_x", "global_y", "global_z"]].to_numpy(float)
            by_gene[g] = (cKDTree(coords), coords)

        # real granules -- the ceiling
        real = sources[sample].copy()
        real["accepted"] = True
        r_pred = A3.dbscan_core_predicate(real, by_gene)
        pred_rows.append(dict(sample=sample, arm="real", d_kind="-", d_label="-",
                              n=len(r_pred), frac_detect=float(r_pred["would_detect"].mean()),
                              median_n_local=float(r_pred["n_local"].median())))

        # tissue-wide random -- the floor / asymptote
        mask, xb, yb = masks[sample]
        rng = np.random.default_rng(SEED)
        rnd = real.copy()
        rx, ry, ok = np.zeros(len(rnd)), np.zeros(len(rnd)), np.zeros(len(rnd), bool)
        for _ in range(20):
            todo = ~ok
            if not todo.any():
                break
            cx = rng.uniform(xb[0], xb[-1], todo.sum())
            cy = rng.uniform(yb[0], yb[-1], todo.sum())
            good = A3.in_tissue(cx, cy, mask, xb, yb)
            idx = np.flatnonzero(todo)[good]
            rx[idx], ry[idx] = cx[good], cy[good]
            ok[idx] = True
        rnd["sphere_x"], rnd["sphere_y"], rnd["accepted"] = rx, ry, ok
        q_pred = A3.dbscan_core_predicate(rnd, by_gene)
        pred_rows.append(dict(sample=sample, arm="random_tissue", d_kind="-", d_label="-",
                              n=int(ok.sum()), frac_detect=float(q_pred["would_detect"][ok].mean()),
                              median_n_local=float(q_pred["n_local"][ok].median())))

        # the vicinity arms -- computed ONCE here and written back onto `pseudo`, so section 5
        # can stratify without refitting (it previously recomputed the whole thing).
        m_s = (pseudo["sample"] == sample) & pseudo["accepted"]
        if m_s.any():
            pp = A3.dbscan_core_predicate(pseudo.loc[m_s], by_gene)
            pseudo.loc[m_s, "would_detect"] = pp["would_detect"].to_numpy()
            pseudo.loc[m_s, "n_local"] = pp["n_local"].to_numpy()
            pseudo.loc[m_s, "gene_missing"] = pp["gene_missing"].to_numpy()
        for (kind, label, arm), grp in pseudo[m_s].groupby(
                ["d_kind", "d_label", "arm"], observed=True):
            pred_rows.append(dict(sample=sample, arm=arm, d_kind=kind, d_label=label,
                                  n=len(grp), frac_detect=float(grp["would_detect"].mean()),
                                  median_n_local=float(grp["n_local"].median()),
                                  n_gene_missing=int(grp["gene_missing"].sum())))
            print(f"[{sample}] {arm:<11} d={kind}:{label:<5} "
                  f"would_detect {grp['would_detect'].mean():.4f}")
        del tx

    pred = pd.DataFrame(pred_rows)
    pred.to_csv(OUT / "detection_predicate.csv", index=False)
    display(pred)

## 5. The distance curve, stratified and thinned

**The shape is the answer.** If the fraction of pseudo-granules that would have been detected rises
sharply as `d` falls toward the granule, the call has genuine local specificity. If it is flat from
5 um out to 50 um -- i.e. already at the tissue-wide asymptote right next to a real granule -- then
it does not, and that would be the reviewer's hypothesis confirmed.

Two controls on the reading:

* **Stratified.** The same curve within each local-density quintile. A difference that survives
  within stratum cannot be a density difference, which is the mechanism the reviewer proposes.
* **Thinned.** 681K pseudo-granules drawn from 681K mutually overlapping sources are not
  independent, so a paired p-value across them would be meaningless. For anything inferential, one
  granule per 25 um spot is retained and effect sizes are reported rather than p-values.

In [ ]:
if RUN_PREDICATE:
    # Consumes the `would_detect` column computed in section 4 -- refitting it here would double
    # the single most expensive step in the notebook for no new information.
    strat_rows, thin_rows = [], []
    acc_all = pseudo[pseudo["accepted"]]

    for (sample, kind, label, arm), acc in acc_all.groupby(
            ["sample", "d_kind", "d_label", "arm"], observed=True):
        for q, sub in acc.groupby("density_quintile", observed=True):
            strat_rows.append(dict(sample=sample, arm=arm, d_kind=kind, d_label=label,
                                   density_quintile=int(q), n=len(sub),
                                   frac_detect=float(sub["would_detect"].mean())))
        for area, sub in acc.groupby("brain_area", observed=True):
            strat_rows.append(dict(sample=sample, arm=arm, d_kind=kind, d_label=label,
                                   brain_area=area, n=len(sub),
                                   frac_detect=float(sub["would_detect"].mean())))

        # thinned: one per VICINITY_THIN_GRID spot, for anything inferential
        keep = (acc.assign(_gx=np.floor(acc["sphere_x"] / C.VICINITY_THIN_GRID).astype(int),
                           _gy=np.floor(acc["sphere_y"] / C.VICINITY_THIN_GRID).astype(int))
                .sample(frac=1.0, random_state=C.VICINITY_THIN_SEED)
                .drop_duplicates(["_gx", "_gy"]))
        thin_rows.append(dict(sample=sample, arm=arm, d_kind=kind, d_label=label,
                              n_thinned=len(keep),
                              frac_detect=float(keep["would_detect"].mean())))

    pd.DataFrame(strat_rows).to_csv(OUT / "detection_predicate_stratified.csv", index=False)
    pd.DataFrame(thin_rows).to_csv(OUT / "detection_predicate_thinned.csv", index=False)
    display(pd.DataFrame(thin_rows))

## 6. The zero-placement-bias variant

The literal vicinity control has to invent sphere positions, and any placement rule is something a
reviewer can argue with. This variant has no placement rule at all.

`all_granules.parquet` is mcDETECT's **rough pass** -- every candidate aggregate, with the size,
in-soma and NC filters all switched off. It is therefore already the table of aggregates that the
pipeline saw and *rejected*, ambient-driven ones included, at real positions found by the real
detector. Comparing Set 2 against `all_granules \ Set 2` as a function of distance from each Set-2
granule asks the same question -- is nearby non-granule space different? -- with the same geometry
and the same detection machinery, and nothing synthesised.

Run **in addition** to the literal control, not instead of it: the reviewer asked for the literal
one by name.

In [ ]:
if RUN_ROUGH_VARIANT:
    rough_rows = []
    for sample in C.SAMPLES:
        rough = pd.read_parquet(C.mcdetect_all_granules_path(sample))
        fine = pd.read_parquet(C.mcdetect_granules_path(sample))

        # rejected candidates = rough spheres with no fine counterpart at the same centre
        ft = cKDTree(fine[["sphere_x", "sphere_y", "layer_z"]].to_numpy(float))
        dmin, _ = ft.query(rough[["sphere_x", "sphere_y", "layer_z"]].to_numpy(float), k=1)
        rejected = rough[dmin > 1e-6].reset_index(drop=True)

        # distance from each rejected candidate to the nearest ACCEPTED granule
        d_to_fine, _ = ft.query(rejected[["sphere_x", "sphere_y", "layer_z"]].to_numpy(float), k=1)
        rejected["d_to_granule"] = d_to_fine

        bins = [0, 2, 5, 10, 20, 50, np.inf]
        rejected["d_bin"] = pd.cut(rejected["d_to_granule"], bins)
        agg = (rejected.groupby("d_bin", observed=True)
               .agg(n=("sphere_r", "size"), median_r=("sphere_r", "median"),
                    median_size=("size", "median"),
                    mean_in_soma=("in_soma_ratio", "mean"))
               .reset_index())
        agg["sample"] = sample
        agg["n_rough"], agg["n_fine"], agg["n_rejected"] = len(rough), len(fine), len(rejected)
        rough_rows.append(agg)
        print(f"[{sample}] rough {len(rough):,} | fine {len(fine):,} | "
              f"rejected {len(rejected):,}")

    rv = pd.concat(rough_rows, ignore_index=True)
    rv["d_bin"] = rv["d_bin"].astype(str)
    rv.to_csv(OUT / "rough_variant_by_distance.csv", index=False)
    display(rv)

## 7. Correctness gates

Off by default. These check the construction, not the biology -- if any fails, the numbers above
are not measuring what the section headings claim.

In [ ]:
if VALIDATE:
    acc = pseudo[pseudo["accepted"]]

    # every accepted offset is on the layer_z grid
    assert set(np.round(acc["layer_z"].unique(), 6)).issubset(set(C.Z_GRID)), "off-grid layer_z"

    # every accepted offset is in tissue
    for sample in C.SAMPLES:
        mask, xb, yb = masks[sample]
        a = acc[acc["sample"] == sample]
        assert A3.in_tissue(a["sphere_x"].to_numpy(), a["sphere_y"].to_numpy(),
                            mask, xb, yb).all(), f"{sample}: offset outside tissue"

    # every accepted offset is at the intended in-plane distance from its source.
    # Paired via `src_i`, NOT the frame index: concat renumbers 0..N and the groupby pools both
    # arms, so the previous index-based pairing never matched and the assert never ran.
    n_checked = 0
    for (sample, kind, label, arm), grp in acc.groupby(
            ["sample", "d_kind", "d_label", "arm"], observed=True):
        src = sources[sample].iloc[grp["src_i"].to_numpy()]
        dd = np.hypot(grp["sphere_x"].to_numpy() - src["sphere_x"].to_numpy(),
                      grp["sphere_y"].to_numpy() - src["sphere_y"].to_numpy())
        assert np.allclose(dd, grp["offset_d"].to_numpy(), atol=1e-6), \
            f"{sample} {arm} {kind}:{label}: offset distance wrong"
        n_checked += len(grp)
    assert n_checked > 0, "the distance gate matched nothing -- it is not actually running"
    print(f"[ok] offset distance verified on {n_checked:,} pseudo-granules")

    # no accepted offset sits inside a nucleus (both arms are documented to reject this)
    for sample in C.SAMPLES:
        a = acc[acc["sample"] == sample]
        hit = np.asarray(nuc[sample].query_ball_point(
            a[["sphere_x", "sphere_y", "layer_z"]].to_numpy(float),
            a["sphere_r"].to_numpy(float), workers=-1, return_length=True))
        assert (hit == 0).all(), f"{sample}: {int((hit>0).sum())} offsets sit inside a nucleus"
    print("[ok] no accepted offset is in a nucleus")

    # radius is matched exactly -- otherwise the count comparison is not even descriptive
    for sample in C.SAMPLES:
        src, a = sources[sample], acc[acc["sample"] == sample]
        assert set(np.round(a["sphere_r"], 9)).issubset(set(np.round(src["sphere_r"], 9)))

    # the rejected arm must actually reject: no accepted offset may overlap a real granule
    for sample in C.SAMPLES:
        a = acc[(acc["sample"] == sample) & (acc["arm"] == "rejected")]
        if len(a):
            hit, _ = A3.overlap_pairs(a, sources[sample], criterion="merge", z_col="layer_z")
            assert not hit.any(), f"{sample}: rejected arm kept an overlapping offset"

    # the real granules must clear their own predicate -- if they do not, the predicate is wrong
    if RUN_PREDICATE:
        pred = pd.read_csv(OUT / "detection_predicate.csv")
        real = pred[pred["arm"] == "real"]
        assert (real["frac_detect"] > 0.95).all(), \
            f"real granules fail their own detection predicate: {real['frac_detect'].tolist()}"

    print("[ok] all gates passed")

## Outputs

| file | contents |
|---|---|
| `source_summary.csv` | n source granules per sample, the seed-gene provenance, and the median NN distance of the seed-gene join |
| `placement_status.csv` | acceptance rate and mean retries per (sample, arm, offset) |
| `vicinity_overlap_with_real.csv` | fraction of unrejected offsets landing on a real granule -- a result, not a nuisance |
| `profile_summary.csv` + `profile_histogram.parquet` | real vs pseudo distributions of `n_total`, `n_marker`, in-nucleus and NC ratio |
| `profile_funnel.csv` | the same filter cascade applied to both sets, stage by stage |
| `detection_predicate.csv` | **the load-bearing table** -- fraction that would have been detected, with the real ceiling and the tissue-wide random floor |
| `detection_predicate_stratified.csv` | the same, within local-density quintile and within brain area |
| `detection_predicate_thinned.csv` | one granule per 25 um spot, for anything inferential |
| `rough_variant_by_distance.csv` | rejected rough-pass candidates by distance to the nearest accepted granule -- the no-placement-rule variant |